<a href="https://colab.research.google.com/github/SodisettiRakesh123/todo-backend/blob/main/Arvyax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import sys
!{sys.executable} -m pip install odfpy openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 29.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for odfpy: filename=odfpy-1.4.1-py2.py3-none-any.whl size=160673 sha256=e58ae28b61dd03dbf7eb7b23693db2ae91811c54896e61f9b913fdc10cbdd2bd
  Stored in directory: /root/.cache/pip/wheels/36/5d/63/8243a7ee78fff0f944d638fd0e66d7278888f5e2285d7346b6
Successfully built odfpy


In [9]:
import pandas as pd

try:
    # Read the ODS file into a pandas DataFrame
    ods_file_path = '/content/Sample_arvyax_reflective_dataset.xlsx.ods'
    df_ods = pd.read_excel(ods_file_path, engine='odf')
    print(f"Successfully loaded {ods_file_path}")

    # Save the DataFrame as training_data.csv
    df_ods.to_csv('training_data.csv', index=False)
    print("Converted ODS file to 'training_data.csv' successfully.")
except FileNotFoundError:
    print(f"Error: The file {ods_file_path} was not found. Please ensure it is uploaded to '/content/'.")
except Exception as e:
    print(f"An error occurred during file conversion: {e}")

Successfully loaded /content/Sample_arvyax_reflective_dataset.xlsx.ods
Converted ODS file to 'training_data.csv' successfully.


In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from textblob import TextBlob
import nltk
nltk.download('punkt', quiet=True)
import warnings
warnings.filterwarnings('ignore')

# Step 1: Load data
try:
    # Google Sheet URL for training data (replace with yours if different)
    # train_url = 'https://docs.google.com/spreadsheets/d/1yLDum7yWr3IH0KivluCBEvqHGlfvFW_S/export?format=csv&gid=0' # gid=0 is usually the first sheet
    # train_df = pd.read_csv(train_url)
    # print("Training data loaded successfully from Google Sheet.")
    print("Attempting to read from local 'training_data.csv'...")
    train_df = pd.read_csv('training_data.csv')
    print("Training data loaded successfully from local file.")
except Exception as e:
    print(f"Error loading training data: {e}")
    print("Please ensure 'training_data.csv' is uploaded to your Colab environment.")
    raise # Re-raise the exception to stop execution and indicate failure

# User requested to use training data as test data
# IMPORTANT: This is generally not recommended for evaluating true generalization performance due to overfitting.
# For a realistic assessment, always use a separate, unseen test dataset.
# Here, we are assigning train_df to test_df as per user's explicit request.
test_df = train_df.copy()
print("Using training data as test data for prediction as requested by the user.")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print(train_df.columns.tolist())  # Check columns

# Step 2: Feature Engineering
def extract_text_features(text):
    if pd.isna(text):
        return {'sentiment': 0.0, 'subjectivity': 0.5, 'length': 0, 'word_count': 0}
    blob = TextBlob(str(text))
    return {
        'sentiment': blob.sentiment.polarity,
        'subjectivity': blob.sentiment.subjectivity,
        'length': len(str(text)),
        'word_count': len(str(text).split())
    }

# Apply text features (only to train_df for now)
text_feats_train = train_df['journal_text'].apply(extract_text_features).apply(pd.Series)
train_df = pd.concat([train_df, text_feats_train], axis=1)
numeric_feats = ['duration_min', 'sleep_hours', 'energy_level', 'stress_level',
                     'sentiment', 'subjectivity', 'length', 'word_count']
train_df[numeric_feats] = train_df[numeric_feats].fillna(train_df[numeric_feats].mean()) # Fill NaN in numeric features

# Categorical encoders
le_state = LabelEncoder()
train_df['emotional_state_encoded'] = le_state.fit_transform(train_df['emotional_state'])
le_intensity = LabelEncoder()  # Though numeric, treat as class for consistency
train_df['intensity_encoded'] = le_intensity.fit_transform(train_df['intensity'].astype(str))

# TF-IDF for text
vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
X_text_train = vectorizer.fit_transform(train_df['journal_text'].fillna('')).toarray()

# Metadata features
# Identify categorical and numerical features for the ColumnTransformer
categorical_features = ['ambience_type', 'time_of_day', 'previous_day_mood', 'face_emotion_hint', 'reflection_quality']
numerical_features = ['duration_min', 'sleep_hours', 'energy_level', 'stress_level',
                      'sentiment', 'subjectivity', 'length', 'word_count']

# Create a ColumnTransformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Apply the preprocessing to the training data
# Fill NaN values in categorical features with 'missing' before one-hot encoding
train_df[categorical_features] = train_df[categorical_features].fillna('missing')

X_meta_train = preprocessor.fit_transform(train_df[numerical_features + categorical_features])

# Combined features
X_train = np.hstack([X_text_train, X_meta_train])
y_state = train_df['emotional_state_encoded']
y_intensity = train_df['intensity_encoded']

# Step 3: Models (Part 1 & 2: Classification for both state & intensity)
X_tr, X_val, y_state_tr, y_state_val, y_int_tr, y_int_val = train_test_split(
    X_train, y_state, y_intensity, test_size=0.2, random_state=42)

# Multi-output XGBoost (lightweight, handles multi-label)
model_state = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
model_intensity = XGBClassifier(n_estimators=100, random_state=42, objective='reg:squarederror')  # Regression-like for intensity

model_state.fit(X_tr, y_state_tr)
model_intensity.fit(X_tr, y_int_tr)

# Validation
state_pred_val = model_state.predict(X_val)
int_pred_val = model_intensity.predict(X_val).round().astype(int)
print("State Accuracy:", accuracy_score(y_state_val, state_pred_val))
print(classification_report(y_state_val, state_pred_val))

# Part 3: Decision Engine (rule-based on preds + context, product-oriented)
def decision_engine(pred_state, pred_int, stress, energy, time_of_day):
    state_name = le_state.inverse_transform([pred_state])[0]
    intensity = int(pred_int)

    if 'anxious' in state_name.lower() or stress > 7:
        what = 'box_breathing'
        when = 'now' if energy > 3 else 'tonight'
    elif pred_int > 3 and energy < 4:
        what = 'rest'
        when = 'now'
    elif time_of_day in ['morning', 'afternoon'] and 'restless' in state_name.lower():
        what = 'movement'
        when = 'within_15_min'
    elif 'sad' in state_name.lower():
        what = 'journaling'
        when = 'later_today'
    else:
        what = 'deep_work'
        when = 'now'

    return what, when

# Part 4: Uncertainty (simple: low conf if short text or high variance)
def get_uncertainty(model, X_sample):
    pred_proba = model.predict_proba(X_sample)
    conf = np.max(pred_proba, axis=1)[0]
    uncertain = 1 if conf < 0.7 or X_sample[0, -1] < 10 else 0  # Short text flag
    return conf, uncertain

# Step 5: Predict on test
# Process test_df similarly to train_df
text_feats_test = test_df['journal_text'].apply(extract_text_features).apply(pd.Series)
test_df = pd.concat([test_df.reset_index(drop=True), text_feats_test], axis=1)
test_df[numeric_feats] = test_df[numeric_feats].fillna(train_df[numeric_feats].mean()) # Fill NaN in numeric features using train mean
test_df[categorical_features] = test_df[categorical_features].fillna('missing') # Fill NaN in categorical features

X_text_test = vectorizer.transform(test_df['journal_text'].fillna('')).toarray()
X_meta_test = preprocessor.transform(test_df[numerical_features + categorical_features]) # Use preprocessor fitted on train
X_test = np.hstack([X_text_test, X_meta_test])

test_df['predicted_state_encoded'] = model_state.predict(X_test)
test_df['predicted_state'] = le_state.inverse_transform(test_df['predicted_state_encoded'])
test_df['predicted_intensity_encoded'] = model_intensity.predict(X_test).round().astype(int)
test_df['predicted_intensity'] = test_df['predicted_intensity_encoded'].astype(str)  # Map back if needed

# Add decisions & uncertainty
predictions = []
for idx, row in test_df.iterrows():
    pred_state = row['predicted_state_encoded']
    pred_int = row['predicted_intensity_encoded']
    # Need to pass correct stress, energy, time_of_day from preprocessed test_df
    # For now, using placeholders, adjust as needed with test_df columns
    stress_val = row['stress_level'] if 'stress_level' in row else 5 # Placeholder
    energy_val = row['energy_level'] if 'energy_level' in row else 5 # Placeholder
    time_of_day_val = row['time_of_day'] if 'time_of_day' in row else 'morning' # Placeholder
    conf, unc = get_uncertainty(model_state, X_test[idx:idx+1])
    what, when = decision_engine(pred_state, pred_int, stress_val, energy_val, time_of_day_val)
    predictions.append({
        'id': row['id'],
        'predicted_state': row['predicted_state'],
        'predicted_intensity': pred_int,
        'confidence': conf,
        'uncertain_flag': unc,
        'what_to_do': what,
        'when_to_do': when
    })

preds_df = pd.DataFrame(predictions)
preds_df.to_csv('predictions.csv', index=False)
print("Predictions saved to predictions.csv")
print(preds_df.head())


Attempting to read from local 'training_data.csv'...
Training data loaded successfully from local file.
Using training data as test data for prediction as requested by the user.
Train shape: (1200, 13)
Test shape: (1200, 13)
['id', 'journal_text', 'ambience_type', 'duration_min', 'sleep_hours', 'energy_level', 'stress_level', 'time_of_day', 'previous_day_mood', 'face_emotion_hint', 'reflection_quality', 'emotional_state', 'intensity']
State Accuracy: 0.5791666666666667
              precision    recall  f1-score   support

           0       0.69      0.57      0.62        51
           1       0.62      0.59      0.60        41
           2       0.50      0.53      0.51        36
           3       0.64      0.59      0.61        46
           4       0.56      0.59      0.58        37
           5       0.45      0.62      0.52        29

    accuracy                           0.58       240
   macro avg       0.58      0.58      0.58       240
weighted avg       0.59      0.58     

In [13]:
import pandas as pd

try:
    # Read the ODS test file into a pandas DataFrame
    ods_test_file_path = '/content/arvyax_test_inputs_120.xlsx.ods'
    df_ods_test = pd.read_excel(ods_test_file_path, engine='odf')
    print(f"Successfully loaded {ods_test_file_path}")

    # Save the DataFrame as test_data.csv
    df_ods_test.to_csv('test_data.csv', index=False)
    print("Converted ODS test file to 'test_data.csv' successfully.")
except FileNotFoundError:
    print(f"Error: The test file {ods_test_file_path} was not found. Please ensure it is uploaded to '/content/'.")
except Exception as e:
    print(f"An error occurred during test file conversion: {e}")

Successfully loaded /content/arvyax_test_inputs_120.xlsx.ods
Converted ODS test file to 'test_data.csv' successfully.
